### Trazenje stacionarnih intervala

In [207]:
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline


In [208]:
file_path = "../../../../results/2018/24/calorimeter_24_year_2018filtriranje.csv"
file_path1="../../../../results/2018/24/zones_24_year_2018filtriranje.csv"

data = pd.read_csv(file_path, index_col=[0])
data1=pd.read_csv(file_path1, index_col=[0])

### Odredivanje intervala i odredivanje njegove duljine

In [210]:
import pandas as pd

# Pretvorba u datetime format i zaokruživanje na minutu
data1['timestamp'] = pd.to_datetime(data1['timestamp']).dt.round('min')

all_intervals = []

for zone_id in range(195, 208):
    # filtriramo za trenutno zone_id
    zone_data = data1[data1['zone_id'] == zone_id]
    
    # Sortiramo po timestampu
    zone_data_sorted = zone_data.sort_values(by='timestamp')
    
    # Grupiramo po promjenama u zone_fan_speed
    zone_data_sorted['change'] = zone_data_sorted['zone_fan_speed'] != zone_data_sorted['zone_fan_speed'].shift()
    zone_data_sorted['group'] = zone_data_sorted['change'].cumsum()
    
    # Združimo kako bi dobili intervale
    intervals = zone_data_sorted.groupby('group').agg(
        start_time=('timestamp', 'first'),
        end_time=('timestamp', 'last'),
        speed=('zone_fan_speed', 'first')   # <- ovdje spremamo fan speed koji se nije mijenjao
    ).reset_index(drop=True)
    
    # Računamo trajanje intervala
    intervals['duration'] = (intervals['end_time'] - intervals['start_time']).dt.total_seconds() / 3600
    
    # Dodajemo zone_id
    intervals['zone_id'] = zone_id
    
    # Dodajemo u listu intervala
    all_intervals.append(intervals)

# Pretvaramo u dataframe
all_intervals_df = pd.concat(all_intervals, ignore_index=True)
print(all_intervals_df)
print("Gotovo")


               start_time            end_time  speed   duration  zone_id
0     2018-03-22 13:42:00 2018-03-23 03:27:00    0.0  13.750000      195
1     2018-03-23 03:28:00 2018-03-23 04:56:00   33.0   1.466667      195
2     2018-03-23 04:57:00 2018-03-23 04:57:00    0.0   0.000000      195
3     2018-03-23 04:58:00 2018-03-23 04:58:00   33.0   0.000000      195
4     2018-03-23 04:59:00 2018-03-23 04:59:00    0.0   0.000000      195
...                   ...                 ...    ...        ...      ...
53083 2018-12-29 16:55:00 2018-12-29 17:00:00   33.0   0.083333      207
53084 2018-12-29 17:01:00 2018-12-31 05:31:00    0.0  36.500000      207
53085 2018-12-31 05:32:00 2018-12-31 05:38:00   33.0   0.100000      207
53086 2018-12-31 05:39:00 2018-12-31 05:39:00   66.5   0.000000      207
53087 2018-12-31 20:02:00 2018-12-31 23:59:00    0.0   3.950000      207

[53088 rows x 5 columns]
Gotovo


### Uklanjanje duplikata i filtriranje duljih od 15min


In [212]:
# stvaramo dataframe od all_intervals
all_intervals_df = pd.concat(all_intervals, ignore_index=True)

# pretvorba u datetime format
all_intervals_df['start_time'] = pd.to_datetime(all_intervals_df['start_time'])
all_intervals_df['end_time'] = pd.to_datetime(all_intervals_df['end_time'])

# filtriranje intervala duljih od 15 min (promijenjeno sa 1h na 15 min)
all_intervals_df['duration'] = all_intervals_df['end_time'] - all_intervals_df['start_time']
filtered_intervals_df = all_intervals_df[all_intervals_df['duration'] >= pd.Timedelta(minutes=15)]

# micemo duplikate
unique_intervals = filtered_intervals_df[['start_time', 'end_time', 'zone_id', 'speed']].drop_duplicates()

print("Gotovo")
print(unique_intervals)

Gotovo
               start_time            end_time  zone_id  speed
0     2018-03-22 13:42:00 2018-03-23 03:27:00      195    0.0
1     2018-03-23 03:28:00 2018-03-23 04:56:00      195   33.0
50    2018-03-23 07:15:00 2018-03-23 07:30:00      195    0.0
52    2018-03-23 07:32:00 2018-03-23 07:51:00      195    0.0
54    2018-03-23 07:53:00 2018-03-23 08:19:00      195    0.0
...                   ...                 ...      ...    ...
53071 2018-12-29 15:19:00 2018-12-29 15:36:00      207   33.0
53077 2018-12-29 16:11:00 2018-12-29 16:26:00      207   33.0
53081 2018-12-29 16:37:00 2018-12-29 16:52:00      207   33.0
53084 2018-12-29 17:01:00 2018-12-31 05:31:00      207    0.0
53087 2018-12-31 20:02:00 2018-12-31 23:59:00      207    0.0

[7446 rows x 4 columns]


### Trazenje presjeka kod intervala

In [214]:
intersections = []


for _, interval in unique_intervals.iterrows():
    start = interval['start_time']
    end = interval['end_time']
    current_zone = interval['zone_id']
    speed= interval['speed']
    
    # gledmo ima li interval presjek u drugim zonama
    overlapping_zones = filtered_intervals_df[
        (filtered_intervals_df['zone_id'] != current_zone) &  # Exclude current zone
        (filtered_intervals_df['start_time'] <= end) & 
        (filtered_intervals_df['end_time'] >= start)
    ]
    
    # provjera ima li presjek u SVIM zonama
    covered_zones = set(overlapping_zones['zone_id'])
    required_zones = set(range(197, 208)) - {current_zone} 

    if required_zones.issubset(covered_zones):  # ako je presjek svih zona dodaj u tablicu

        intersection_start = max(overlapping_zones['start_time'].min(), start)
        intersection_end = min(overlapping_zones['end_time'].max(), end)
        intersection_duration = intersection_end - intersection_start

        # pohrani rezultat
        intersections.append({
            'zone_id': current_zone,  # Store the reference zone
            'intersection_start': intersection_start,
            'intersection_end': intersection_end,
            'intersection_duration': intersection_duration, 
            'speed':speed
        })


intersections_df = pd.DataFrame(intersections)
print(intersections_df.head())

intersections_df.to_csv('../../../../results/2018/24/15min/calorimeter_24_year_2018presjeci.csv ', index=False)

print("Gotovo'")

   zone_id  intersection_start    intersection_end intersection_duration  \
0      195 2018-03-22 13:42:00 2018-03-23 03:27:00       0 days 13:45:00   
1      195 2018-03-23 03:28:00 2018-03-23 04:56:00       0 days 01:28:00   
2      195 2018-03-23 17:55:00 2018-03-24 04:27:00       0 days 10:32:00   
3      195 2018-03-24 07:12:00 2018-03-24 09:17:00       0 days 02:05:00   
4      195 2018-03-24 09:18:00 2018-03-24 10:24:00       0 days 01:06:00   

   speed  
0    0.0  
1   33.0  
2    0.0  
3    0.0  
4   33.0  
Gotovo'


### Uredivanje tablice za intervale

In [216]:

# Fitracija za zone_id 195
#intersections_df = intersections_df[intersections_df['zone_id'] == 195]
print("Number of rows:", len(intersections_df))
# micem zone_id
intersections_df = intersections_df.drop(columns=['zone_id'])

# Spremanje rezultata
intersections_df.to_csv( '../../../../results/2018/24/15min/calorimeter_24_year_2018presjeci.csv ', index=False)
print("Gotov")
print(intersections_df.head())



Number of rows: 2673
Gotov
   intersection_start    intersection_end intersection_duration  speed
0 2018-03-22 13:42:00 2018-03-23 03:27:00       0 days 13:45:00    0.0
1 2018-03-23 03:28:00 2018-03-23 04:56:00       0 days 01:28:00   33.0
2 2018-03-23 17:55:00 2018-03-24 04:27:00       0 days 10:32:00    0.0
3 2018-03-24 07:12:00 2018-03-24 09:17:00       0 days 02:05:00    0.0
4 2018-03-24 09:18:00 2018-03-24 10:24:00       0 days 01:06:00   33.0


In [226]:
only_end = intersections_df[["intersection_end", "speed"]]
print(only_end.head())

# Save to CSV
only_end.to_csv('../../../../results/2018/24/15min/calorimeter_24_year_2018presjeci.csv', index=False)

     intersection_end  speed
0 2018-03-23 03:27:00    0.0
1 2018-03-23 04:56:00   33.0
2 2018-03-24 04:27:00    0.0
3 2018-03-24 09:17:00    0.0
4 2018-03-24 10:24:00   33.0
